# RoboManipBaselines Colab Tutorial

## Getting Started with Manipulation × AI：模倣学習の体験セッション

このノートブックは、[SIマニピュレーション若手の会](https://sites.google.com/view/sice-si-manipulation/Wakate) がワークショップ用に作成した、Google Colab向けのハンズオン教材です。

題材には、産業技術総合研究所を中心に開発されているオープンソースソフトウェア [RoboManipBaselines](https://github.com/isri-aist/RoboManipBaselines) を使用します。RoboManipBaselinesは、ロボットマニピュレーションにおける模倣学習手法を、実世界およびシミュレーション環境で再現・検証するためのフレームワークです。

> **注意**  
> このノートブックはワークショップ用の非公式教材です。RoboManipBaselinesの開発者および公式リポジトリに、このノートブック固有の内容について問い合わせないでください。

<p align="center">
  <img src="https://www.aist.go.jp/Portals/0/resource_images/aist_j/press_release/pr2025/pr20250123_2/fig2.jpg" alt="RoboManipBaselines overview" width="90%">
</p>

関連リンク:

- [RoboManipBaselines GitHub Repository](https://github.com/isri-aist/RoboManipBaselines)
- [RoboManipBaselines Project Page](https://isri-aist.github.io/RoboManipBaselines-ProjectPage/)
- [RoboManipBaselines arXiv](https://arxiv.org/abs/2509.17057)


## 0. Google Colabの設定

このノートブックは、Google ColabのGPUランタイムで実行することを想定しています。

1. 上部メニューから **ランタイム** → **ランタイムのタイプを変更** を開く
2. **ハードウェア アクセラレータ** に **T4 GPU** または利用可能なGPUを選択する
3. **保存** を押す
4. 上から順にセルを実行する

Colabの実行環境は更新されるため、最初にPython、GPU、PyTorchの状態を確認します。Colabの計算資源は保証されず、利用状況によりGPUの種類や利用可能時間が変わることがあります。


In [ ]:
import os
import platform
import subprocess
import sys

print("Python:", sys.version)
print("Platform:", platform.platform())
print("Working directory:", os.getcwd())

try:
    gpu_info = subprocess.check_output(["nvidia-smi"], text=True)
    print(gpu_info)
except Exception as exc:
    print("GPUを確認できませんでした。ランタイムのハードウェアアクセラレータをGPUに変更してください。")
    print(type(exc).__name__, exc)


## 1. RoboManipBaselinesのインストール

RoboManipBaselines本体とサブモジュールを `/content/RoboManipBaselines` に取得し、Colab上で編集可能モードとしてインストールします。

このセルは再実行できるように、既にリポジトリが存在する場合は `git pull` と `git submodule update` を実行します。Colabの標準環境に入っているPyTorchの版は変わることがあるため、先にCUDA 12.4向けのPyTorch 2.6系とTorchCodec 0.2系に揃えてから、RoboManipBaselinesをインストールします。


In [ ]:
%%bash
set -euxo pipefail

cd /content

if [ ! -d RoboManipBaselines/.git ]; then
  git clone https://github.com/isri-aist/RoboManipBaselines.git --recursive
else
  cd /content/RoboManipBaselines
  git pull --ff-only
  git submodule update --init --recursive
fi

apt-get update -qq
apt-get install -y -qq ffmpeg

python -m pip install --upgrade pip setuptools wheel
python -m pip uninstall -y torch torchvision torchaudio torchcodec || true
python -m pip install \
  torch==2.6.0+cu124 \
  torchvision==0.21.0+cu124 \
  torchaudio==2.6.0+cu124 \
  --index-url https://download.pytorch.org/whl/cu124
python -m pip install torchcodec==0.2.0

cd /content/RoboManipBaselines
python -m pip install -e .


## 2. ACT用の追加依存関係をインストール

このチュートリアルでは、模倣学習手法の一つである **Action Chunking with Transformer (ACT)** を使います。ACTを動かすために、RoboManipBaselinesの追加依存関係と、ACTサブモジュール内の `detr` をインストールします。


In [ ]:
%%bash
set -euxo pipefail

cd /content/RoboManipBaselines
python -m pip install -e ".[act]"

cd /content/RoboManipBaselines/third_party/act/detr
python -m pip install -e .


## 3. インストール結果を確認

Pythonから主要ライブラリを読み込み、GPU版のPyTorchが利用できるかを確認します。`torch.cuda.is_available()` が `True` になっていれば、GPUを使う準備ができています。


In [ ]:
import torch

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device:", torch.cuda.get_device_name(0))
    print("CUDA version used by PyTorch:", torch.version.cuda)


## 4. サンプルデータセットのダウンロード

学習の流れを体験するために、`MujocoUR5eCable_Dataset30` をダウンロードします。このデータセットには、`MujocoUR5eCable` 環境のデモンストレーションデータが含まれています。

データセットは `/content/RoboManipBaselines/robo_manip_baselines/dataset/MujocoUR5eCable` に展開します。再実行時に古い展開結果が残らないよう、展開先を一度削除してから作成します。


In [ ]:
%%bash
set -euxo pipefail

DATASET_ROOT=/content/RoboManipBaselines/robo_manip_baselines/dataset
DATASET_ZIP=MujocoUR5eCable_Dataset30.zip
DATASET_DIR=MujocoUR5eCable
DATASET_URL="https://www.dropbox.com/scl/fo/sykc20cnax2scom1u8sc6/AM-zLM8dAZ5h6EQ8eDXcZic?rlkey=7icbmjc6wdqnp0tngfjqlhwoh&dl=1"

cd "${DATASET_ROOT}"
rm -rf "${DATASET_DIR}"
mkdir -p "${DATASET_DIR}"

curl -L -o "${DATASET_ZIP}" "${DATASET_URL}"
unzip -q -o "${DATASET_ZIP}" -d "${DATASET_DIR}"
rm -f "${DATASET_ZIP}"

du -sh "${DATASET_DIR}"
find "${DATASET_DIR}" -maxdepth 2 -type f | head


## 5. ACTポリシーを学習

ダウンロードしたデータセットを使って、ACTポリシーの学習を実行します。

学習には時間がかかります。ワークショップ中は、ログが流れ始め、損失値や保存先が確認できれば、必要に応じて途中で停止して構いません。


In [ ]:
%cd /content/RoboManipBaselines/robo_manip_baselines
!python ./bin/Train.py Act --dataset_dir ./dataset/MujocoUR5eCable --batch_size 32


## 6. 次に確認すること

学習が開始できたら、以下を確認します。

- 学習ログに表示されるlossの推移
- 学習済みモデルやログの保存先
- データセットの件数、画像、ロボット状態、行動の対応関係
- 実機やシミュレーションで使う場合に追加で必要になる設定

Colabのセッションは一定時間で切断されることがあります。必要な結果はGoogle Driveやローカル環境に保存してください。
